# 🤖 Notebook 1 — Il Mio Primo Prompt a un'AI Locale

In questo notebook imparerai a:
1. Caricare un modello AI che gira **sul computer locale** (nessun dato inviato a server esterni)
2. Scrivere un **prompt** e ricevere una risposta
3. Capire cosa sono i **token** e come funziona la generazione di testo

---
## 🔧 Passo 1: Importa OpenVINO GenAI

In [ ]:
import openvino_genai as ov_genai
from pathlib import Path
import time

print(f"OpenVINO GenAI versione: {ov_genai.__version__}")
print("✅ Importazione riuscita!")

## 🔧 Passo 2: Carica il Modello

Usiamo `LLMPipeline`: è l'oggetto principale di OpenVINO GenAI che:
- Carica il modello in memoria RAM
- Lo ottimizza per la CPU (o GPU Intel)
- Espone un metodo `.generate()` per generare testo

In [ ]:
MODEL_DIR = "/workspace/models/tinyllama-chat-ov"
DEVICE = "CPU"   # prova "GPU" se hai una GPU Intel

print(f"Caricamento modello da: {MODEL_DIR}")
print(f"Dispositivo: {DEVICE}")
print("(può richiedere 10-30 secondi la prima volta...)")

t0 = time.time()
pipe = ov_genai.LLMPipeline(MODEL_DIR, DEVICE)
print(f"\n✅ Modello caricato in {time.time()-t0:.1f}s")

## ✏️ Passo 3: Scrivi il Tuo Primo Prompt

Modifica la variabile `mio_prompt` con la domanda che vuoi fare all'AI!

In [ ]:
# ✏️ MODIFICA QUI il tuo prompt!
mio_prompt = "Spiega in 3 frasi cos'è l'intelligenza artificiale."

# Formato ChatML — il 'codice segreto' che capisce TinyLlama
prompt_formattato = f"""<|system|>
Sei un assistente utile. Rispondi sempre in italiano, in modo chiaro e conciso.</s>
<|user|>
{mio_prompt}</s>
<|assistant|>
"""

print("Prompt che verrà inviato al modello:")
print("─" * 40)
print(prompt_formattato)
print("─" * 40)

## 🚀 Passo 4: Genera la Risposta

Usiamo lo **streaming**: ogni parola appare non appena viene generata, proprio come ChatGPT.

In [ ]:
import sys

# Parametri di generazione
config = ov_genai.GenerationConfig()
config.max_new_tokens = 200   # quante parole (circa) generare
config.temperature    = 0.7   # 0 = sempre uguale, 1 = molto creativo
config.top_p          = 0.9
config.do_sample      = True

# Callback di streaming: chiamato per ogni token generato
tokens_generati = []
def streamer(token: str) -> bool:
    print(token, end="", flush=True)
    tokens_generati.append(token)
    return False  # True = interrompe la generazione

print("🤖 Risposta:\n")
print("─" * 50)
t_start = time.time()

pipe.generate(prompt_formattato, config, streamer)

t_end = time.time()
print("\n" + "─" * 50)
print(f"\n⏱️  {len(tokens_generati)} token in {t_end-t_start:.1f}s  "
      f"({len(tokens_generati)/(t_end-t_start):.1f} token/s)")

## 🧪 Passo 5: Sperimenta!

Prova a cambiare i parametri e osserva come cambia la risposta:

In [ ]:
# Esperimento: confronta temperature diverse
domanda = "Completa questa frase: 'Il cielo è'"

for temp in [0.0, 0.5, 1.0]:
    p = f"<|system|>\nRispondi in italiano.</s>\n<|user|>\n{domanda}</s>\n<|assistant|>\n"
    cfg = ov_genai.GenerationConfig()
    cfg.max_new_tokens = 30
    cfg.temperature = temp
    cfg.do_sample = (temp > 0)
    
    risposta = pipe.generate(p, cfg)
    # Estrai solo il testo generato
    testo = risposta if isinstance(risposta, str) else str(risposta)
    print(f"Temperatura {temp:.1f}: {testo.strip()[:80]}")